# PFB Output Calibration and Linearity Analysis

This notebook explains how to measure, analyze, and calibrate the output of the signal chain
**ADC → PFB** and convert digital FFT results in dBFS into real RF power in dBm.

The `QOUT` parameter is set to **6** for all measurements in this notebook:

```python
chain.analysis.qout(6)
```

After the PFB the channel sampling rate is decimated by 8, so the effective channel rate is:
$$F_{s,\text{ch}} = \frac{4096\,\text{MHz}}{2 \times 8} = 256\,\text{MHz}$$

---
## 1. System Architecture

The complete signal path from RF input to the `x_preddscic` data files is:

```
RF Input
  │
  ▼
ADC  (fs = 4096 MHz, 12-bit, MSB-aligned to 16-bit → FS = 2¹⁵)
  │  Mixer × e^{−j2π(−Fs/4)n}  +  Decimation ÷2
  ▼
axis_pfb_8x16  (16 channels, Decimation = 8, fs_ch = 256 MHz)
  │  Polyphase accumulation × 8  +  qout right-shift ÷ 2^6
  ▼
PFB_CHSEL  (channel selection)
  │
  ▼
x_preddscic_t_data.txt        ← data analysed in this notebook
  │
 [DDS + CIC bypassed]
```

### Hardware blocks relevant to calibration

| Block | Action | Effect on signal amplitude |
|---|---|---|
| ADC (12-bit → 16-bit MSB) | Left shift × 2⁴ | captured in K_ADC |
| Mixer + Decimation ÷2 | IQ down-conversion | captured in K_ADC |
| PFB polyphase accumulation (D=8) | Coherent sum of 8 branches | **+20·log₁₀(8) = +18.06 dB** |
| qout right-shift (qout=6) | Divide amplitude by 2⁶ = 64 | **−20·log₁₀(64) = −36.12 dB** |

> **Key insight — the polyphase accumulation gain.**  
> A polyphase filter bank with D branches coherently accumulates D time-domain samples before
> the FFT stage. For a tonal input this is equivalent to a matched filter: the signal amplitude
> at the output of each polyphase branch grows by a factor D in amplitude (= +20·log₁₀(D) dB).
> The `qout` right-shift then divides the amplitude by 2^qout.  
> The **net PFB gain** is therefore:  
> $$G_{\text{PFB,net}} = +20\log_{10}(D) - 20\log_{10}(2^{q_{\text{out}}})$$
> $$= 20\log_{10}(8) - 20\log_{10}(64) = 18.06 - 36.12 = -18.06\,\text{dB}$$
>
> **This was confirmed empirically:**  
> measuring power in the time domain on both `x_adc` and `x_preddscic` files at the same
> input level gave G_PFB,net = −18.14 dB, in excellent agreement with the model.

---
## 2. The `qout` Parameter

Inside the FPGA the PFB FFT produces a **27-bit full-precision output**. The system output
is **16 bits**, so up to 11 bits must be removed. The VHDL logic:

```vhdl
qout_reg_i <= (others => '0')
    when ( unsigned(QOUT_REG) > to_unsigned(11, QOUT_REG'length) )
    else unsigned(QOUT_REG(3 downto 0));
```

This **saturates** the index to 0 if `QOUT_REG > 11` (range protection) and then applies
an **arithmetic right shift** of `qout_reg_i` bits to the FFT output before truncating to
16 bits. A right shift by N bits divides the amplitude by 2ᴺ:

$$x_{\text{out}}[n] = \frac{x_{\text{fft}}[n]}{2^{q_{\text{out}}}}$$

The resulting amplitude attenuation in dB is:
$$G_{q_{\text{out}}} = -20\log_{10}(2^{q_{\text{out}}}) = -6.02 \cdot q_{\text{out}}$$

For `qout = 6`: $G_{q_{\text{out}}} = -36.12\,\text{dB}$.

| qout | Divisor | G_qout (dB) |
|------|---------|-------------|
| 0 | 1 | 0.00 |
| 3 | 8 | −18.06 |
| 6 | 64 | −36.12 |
| 9 | 512 | −54.19 |
| 11 | 2048 | −66.22 |

---
## 3. Calibration Model

### 3.1 ADC calibration (from Notebook 01)

From the ADC raw-data notebook the empirical relationship is:

$$P_{\text{adc}}\,(\text{dBFS}) = P_{\text{in}}\,(\text{dBm}) + K_{\text{ADC}}$$

where $K_{\text{ADC}} = -12.89\,\text{dB}$ already absorbs every analog loss and
the 12→16-bit MSB alignment, using $\text{FS} = 2^{15}$.

### 3.2 PFB net gain

$$G_{\text{PFB,net}} = \underbrace{+20\log_{10}(D)}_{\text{polyphase gain}} + \underbrace{(-20\log_{10}(2^{q_{\text{out}}}))}_{\text{qout attenuation}}$$

With $D = 8$ and $q_{\text{out}} = 6$:

$$G_{\text{PFB,net}} = +18.06 - 36.12 = -18.06\,\text{dB}$$

### 3.3 Complete chain

$$P_{\text{pfb}}\,(\text{dBFS}) = P_{\text{in}}\,(\text{dBm}) + K_{\text{ADC}} + G_{\text{PFB,net}}$$

Rearranging to recover RF power:

$$\boxed{P_{\text{in}}\,(\text{dBm}) = P_{\text{pfb}}\,(\text{dBFS}) + \underbrace{\left(-K_{\text{ADC}} - G_{\text{PFB,net}}\right)}_{\text{CAL\_CONSTANT}}}$$

$$\text{CAL\_CONSTANT} = -K_{\text{ADC}} - G_{\text{PFB,net}} = 12.89 + 18.06 = 30.95\,\text{dB}$$

### 3.4 Budget summary

| Contribution | Formula | Value |
|---|---|---|
| Analog chain + ADC | K_ADC | −12.89 dB |
| Polyphase accumulation (D=8) | +20·log₁₀(8) | +18.06 dB |
| qout right-shift (qout=6) | −20·log₁₀(64) | −36.12 dB |
| **Net PFB gain** | | **−18.06 dB** |
| **CAL_CONSTANT** | −K_ADC − G_PFB,net | **+30.95 dB** |

> **Previous (incorrect) model** used only the qout term ($-36.12\,\text{dB}$),
> ignoring the polyphase accumulation gain (+18.06 dB).  
> This produced CAL_CONSTANT = 49.01 dB and a systematic error of **~+20 dB**.
> The corrected model reduces the error to **< 0.5 dB**.

---
## 4. Python Implementation

In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
import glob, re

In [ ]:
# ============================================================
# SYSTEM PARAMETERS
# ============================================================
Fs      = 256e6   # PFB channel sample rate (Hz)
FS      = 2**15   # Digital full-scale (12-bit ADC MSB-aligned to 16-bit)
tone_bins = 3     # Half-width of tone integration window (bins)

# ------------------------------------------------------------
# ADC calibration constant (from Notebook 01)
# P_adc(dBFS) = P_in(dBm) + K_ADC
# ------------------------------------------------------------
K_ADC = -12.89  # dB  (negative: signal is attenuated before ADC)

# ------------------------------------------------------------
# PFB parameters
# ------------------------------------------------------------
PFB_DECIMATION = 8   # Number of polyphase branches (from axis_pfb_8x16, Decimation=8)
PFB_QOUT       = 6   # Right-shift applied to FFT output before truncation to 16-bit

# Polyphase accumulation: D branches coherently sum → amplitude gain of D
G_polyphase = 20 * np.log10(PFB_DECIMATION)    # +18.06 dB

# qout right-shift: amplitude divided by 2^qout
G_qout      = -20 * np.log10(2**PFB_QOUT)      # -36.12 dB

# Net PFB amplitude gain
G_PFB_net   = G_polyphase + G_qout             # -18.06 dB

# ------------------------------------------------------------
# Calibration constant
# P_in(dBm) = P_pfb(dBFS) + CAL_CONSTANT
# CAL_CONSTANT = -K_ADC - G_PFB_net
# ------------------------------------------------------------
CAL_CONSTANT = -K_ADC - G_PFB_net              # +30.95 dB

print("=== Signal Chain Parameters ===")
print(f"  FS (digital full-scale):      2^15 = {FS}")
print(f"  K_ADC (from notebook 01):     {K_ADC:.2f} dB")
print()
print("=== PFB Gain Budget ===")
print(f"  Polyphase accumulation (D=8): {G_polyphase:+.2f} dB")
print(f"  qout right-shift (qout=6):    {G_qout:+.2f} dB")
print(f"  Net PFB gain G_PFB,net:       {G_PFB_net:+.2f} dB")
print()
print("=== Calibration ===")
print(f"  CAL_CONSTANT = -K_ADC - G_PFB,net = {CAL_CONSTANT:.2f} dB")
print(f"  Formula: P_in (dBm) = P_pfb (dBFS) + {CAL_CONSTANT:.2f}")

### 4.1 Load data and compute tone power

In [ ]:
# ============================================================
# LOAD AND PROCESS DATA
# ============================================================
data_path = "data/f300_data/f_300*/x_preddscic_t_data.txt"
files = sorted(glob.glob(data_path),
               key=lambda f: float(re.search(r'(-?\+?\d+)dBm', f).group(1)))

measurements = []

print(f"{'Pin (dBm)':>10}  {'P_pfb (dBFS)':>14}  {'P_time (dBFS)':>15}  {'Δ (dB)':>8}")
print("-" * 55)

for file in files:
    Pin_gen = float(re.search(r'(-?\+?\d+)dBm', file).group(1))

    x = np.loadtxt(file, dtype=np.complex128)
    N = len(x)

    # ----------------------------------------------------------
    # Power in time domain (ground truth — no FFT assumptions)
    # P_time = 10*log10( mean(|x|²) / FS² )
    # ----------------------------------------------------------
    P_time = 10 * np.log10(np.mean(np.abs(x)**2) / FS**2)

    # ----------------------------------------------------------
    # Power via windowed FFT  (tone integration)
    # Normalisation: fft/N then correct for Hanning coherent gain
    # This gives amplitude spectrum in counts; sum |mag|² over
    # tone bins to get tone power in counts².
    # ----------------------------------------------------------
    window        = np.hanning(N)
    coherent_gain = np.sum(window) / N   # ≈ 0.5 for Hanning
    xw            = x * window

    fft_complex = np.fft.fftshift(np.fft.fft(xw) / N)
    mag         = np.abs(fft_complex) / coherent_gain   # amplitude spectrum [counts]

    peak_idx  = np.argmax(mag)
    idx0, idx1 = peak_idx - tone_bins, peak_idx + tone_bins + 1
    tone_power  = np.sum(mag[idx0:idx1]**2)             # [counts²]

    P_pfb = 10 * np.log10(tone_power / FS**2)           # dBFS

    measurements.append({
        'Pin_gen': Pin_gen,
        'P_pfb':   P_pfb,
        'P_time':  P_time,
        'file':    file,
    })

    print(f"{Pin_gen:10.1f}  {P_pfb:14.2f}  {P_time:15.2f}  {P_pfb - P_time:8.2f}")

print()
print("Note: P_pfb ≈ P_time is expected for a pure tone (only noise outside tone bins).")
print("Δ > 0 means FFT method captures slightly more power (noise in integration window).")

### 4.2 Verify the PFB gain budget against measured data

In [ ]:
# ============================================================
# EMPIRICAL VERIFICATION OF G_PFB,net
# ============================================================
# Compare time-domain power in x_preddscic vs the ADC model.
# At Pin = -20 dBm:
#   Expected P_adc = Pin + K_ADC = -20 + (-12.89) = -32.89 dBFS
#   Expected P_pfb = P_adc + G_PFB,net = -32.89 + (-18.06) = -50.95 dBFS
# The measured value should match within ~0.5 dB.

print("=== PFB Gain Verification (linear region) ===")
print(f"{'Pin':>6}  {'P_pfb_meas':>12}  {'P_pfb_model':>13}  {'Δ':>6}")
print("-" * 45)

linear_meas = [m for m in measurements if m['Pin_gen'] <= 5]
deltas = []

for m in linear_meas:
    P_pfb_model = m['Pin_gen'] + K_ADC + G_PFB_net
    delta = m['P_time'] - P_pfb_model
    deltas.append(delta)
    print(f"{m['Pin_gen']:6.1f}  {m['P_time']:12.2f}  {P_pfb_model:13.2f}  {delta:6.2f}")

print()
print(f"Mean Δ: {np.mean(deltas):+.3f} dB   (ideal = 0)")
print(f"Std Δ:  {np.std(deltas):.3f} dB")
print()
print("Measured G_PFB,net (from linear fit):")
measured_G = np.mean([m['P_time'] - (m['Pin_gen'] + K_ADC) for m in linear_meas])
print(f"  G_PFB,net (measured) = {measured_G:.2f} dB")
print(f"  G_PFB,net (model)    = {G_PFB_net:.2f} dB")
print(f"  Difference           = {measured_G - G_PFB_net:.2f} dB")

### 4.3 Apply calibration and check recovery accuracy

In [ ]:
# ============================================================
# CALIBRATION APPLICATION AND ERROR ANALYSIS
# ============================================================
results = {
    'Pin_true':      [],
    'Pin_recovered': [],
    'P_pfb':         [],
    'error':         [],
}

print(f"{'File':<30} {'Pin_true':>10} {'P_pfb':>10} {'Pin_rec':>10} {'Error':>8}")
print("=" * 72)

for m in measurements:
    Pin_rec = m['P_pfb'] + CAL_CONSTANT
    error   = Pin_rec - m['Pin_gen']

    results['Pin_true'].append(m['Pin_gen'])
    results['Pin_recovered'].append(Pin_rec)
    results['P_pfb'].append(m['P_pfb'])
    results['error'].append(error)

    fname = m['file'].split('/')[-2]
    print(f"{fname:<30} {m['Pin_gen']:10.1f} {m['P_pfb']:10.2f} {Pin_rec:10.2f} {error:+8.2f}")

# Linear-region statistics
lin_errors = [e for e, p in zip(results['error'], results['Pin_true']) if p <= 5]

print()
print("=== Error Statistics (linear region, Pin ≤ +5 dBm) ===")
print(f"  Mean error:   {np.mean(lin_errors):+.3f} dB")
print(f"  Std dev:       {np.std(lin_errors):.3f} dB")
print(f"  Max |error|:   {np.max(np.abs(lin_errors)):.3f} dB")
print(f"  RMS error:     {np.sqrt(np.mean(np.array(lin_errors)**2)):.3f} dB")

# Empirical linear fit for comparison
lin_idx = [i for i, p in enumerate(results['Pin_true']) if p <= 5]
P_pfb_lin = np.array(results['P_pfb'])[lin_idx]
Pin_lin   = np.array(results['Pin_true'])[lin_idx]
coeffs    = np.polyfit(P_pfb_lin, Pin_lin, 1)
print()
print("=== Empirical linear fit (verification) ===")
print(f"  Pin = {coeffs[0]:.4f} × P_pfb + {coeffs[1]:.2f}")
print(f"  Expected slope:     1.0000   (actual: {coeffs[0]:.4f})")
print(f"  Expected intercept: {CAL_CONSTANT:.2f} dB  (actual: {coeffs[1]:.2f} dB)")
print(f"  Difference:         {abs(coeffs[1] - CAL_CONSTANT):.2f} dB")

### 4.4 Compression analysis

In [ ]:
# ============================================================
# 1-dB COMPRESSION POINT
# ============================================================
# Compute the mean gain in the linear region and find the first
# point where gain has dropped by 1 dB relative to that mean.

linear_gain = np.mean(
    [r - t for r, t in zip(results['Pin_recovered'], results['Pin_true'])
     if t <= 0]
)

print("=== Compression Analysis ===")
print(f"  Linear-region mean offset: {linear_gain:+.3f} dB (ideal = 0)")
print()

for t, r in zip(results['Pin_true'], results['Pin_recovered']):
    gain = r - t
    compression = linear_gain - gain
    if compression >= 1.0:
        print(f"  1-dB compression point: Pin = {t:+.1f} dBm")
        break
else:
    print("  No 1-dB compression detected in measured range.")

### 4.5 Plots

In [ ]:
# ============================================================
# PLOTTING
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('PFB Power Calibration — Corrected Model', fontsize=14, fontweight='bold')

Pin_arr = np.array(results['Pin_true'])
Rec_arr = np.array(results['Pin_recovered'])
Err_arr = np.array(results['error'])
Pfb_arr = np.array(results['P_pfb'])

# ── Plot 1: Recovered vs True ─────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(Pin_arr, Rec_arr, 'o-', ms=8, lw=2, color='steelblue', label='Recovered')
ax.plot(Pin_arr, Pin_arr, 'k--', lw=2, label='Ideal (1:1)')
ax.set_xlabel('True Input Power (dBm)')
ax.set_ylabel('Recovered Power (dBm)')
ax.set_title('Power Recovery')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 2: Error vs Input Power ──────────────────────────────────────────
ax = axes[0, 1]
ax.plot(Pin_arr, Err_arr, 'o-', ms=8, lw=2, color='seagreen')
ax.axhline(0, color='k', ls='--', lw=2)
ax.axhline( 0.5, color='orange', ls=':', lw=1.5, alpha=0.8, label='±0.5 dB')
ax.axhline(-0.5, color='orange', ls=':', lw=1.5, alpha=0.8)
ax.fill_between(Pin_arr, -0.5, 0.5, alpha=0.12, color='seagreen')
ax.set_xlabel('Input Power (dBm)')
ax.set_ylabel('Error (dB)')
ax.set_title(f'Calibration Error  (σ = {np.std(lin_errors):.3f} dB, linear region)')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 3: PFB output level vs Input ────────────────────────────────────
ax = axes[1, 0]
ax.plot(Pin_arr, Pfb_arr, 'o-', ms=8, lw=2, color='mediumpurple', label='Measured')
expected_pfb = Pin_arr + K_ADC + G_PFB_net
ax.plot(Pin_arr, expected_pfb, 'k--', lw=2, alpha=0.6, label='Model (slope = 1)')
ax.set_xlabel('Input Power (dBm)')
ax.set_ylabel('PFB Digital Level (dBFS)')
ax.set_title('PFB Output vs Input Power')
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate model parameters
ax.annotate(
    f'Slope = 1\nOffset = K_ADC + G_PFB,net\n= {K_ADC:.2f} + {G_PFB_net:.2f}\n= {K_ADC+G_PFB_net:.2f} dB',
    xy=(Pin_arr[3], expected_pfb[3]),
    xytext=(20, 30), textcoords='offset points',
    fontsize=9, color='dimgray',
    arrowprops=dict(arrowstyle='->', color='dimgray', lw=0.8)
)

# ── Plot 4: Error histogram ───────────────────────────────────────────────
ax = axes[1, 1]
ax.hist(lin_errors, bins=min(12, len(lin_errors)), edgecolor='k', alpha=0.7, color='seagreen')
ax.axvline(0,                  color='k', ls='--', lw=2, label='Zero error')
ax.axvline(np.mean(lin_errors), color='r', ls='-',  lw=2,
           label=f'Mean = {np.mean(lin_errors):+.3f} dB')
ax.set_xlabel('Error (dB)')
ax.set_ylabel('Count')
ax.set_title('Error Distribution (linear region)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('images/power_calibration_corrected.pdf', dpi=300, bbox_inches='tight')
plt.show()

---
## 5. Summary

In [ ]:
print("=" * 65)
print("CALIBRATION SUMMARY — PFB (x_preddscic, DDS+CIC bypassed)")
print("=" * 65)
print()
print("Signal chain:  RF → ADC → PFB (Decimation=8, qout=6) → data")
print()
print("Gain budget:")
print(f"  K_ADC (analog + ADC):        {K_ADC:+.2f} dB")
print(f"  Polyphase accumulation (D=8):{G_polyphase:+.2f} dB")
print(f"  qout right-shift (qout=6):   {G_qout:+.2f} dB")
print(f"  Net PFB gain G_PFB,net:      {G_PFB_net:+.2f} dB")
print()
print(f"Calibration constant (analytical): {CAL_CONSTANT:.2f} dB")
print(f"Calibration constant (empirical):  {coeffs[1]:.2f} dB")
print(f"Agreement:                         {abs(CAL_CONSTANT - coeffs[1]):.2f} dB")
print()
print(f"Recovery formula:")
print(f"  P_in (dBm) = P_pfb (dBFS) + {CAL_CONSTANT:.2f}")
print()
print(f"Accuracy (linear region):")
print(f"  Mean error:  {np.mean(lin_errors):+.3f} dB")
print(f"  Std dev:      {np.std(lin_errors):.3f} dB")
print(f"  RMS error:    {np.sqrt(np.mean(np.array(lin_errors)**2)):.3f} dB")
print()
print("Common mistake to avoid:")
print("  Using only qout attenuation (−36.12 dB) without the")
print("  polyphase accumulation gain (+18.06 dB) gives")
print("  CAL_CONSTANT = 49.01 dB → systematic error of ~+20 dB.")
print("=" * 65)

---
## 6. Phase Analysis

The phase measured from the FFT of a captured record is:

$$\phi_{\text{measured}} = \phi_0 + 2\pi f_0 t_0$$

where $t_0$ is the (random) acquisition start time. Because the RF generator and ADC clocks
are **not phase-locked**, $t_0$ varies between captures and the measured phase is uniformly
distributed on $[-\pi, \pi]$. This is **expected behaviour** and does not indicate a problem.

The **amplitude** and **frequency** of the tone are unaffected by this; only the absolute
phase is arbitrary. The `qout` scaling does not rotate phase.

Phase measurements are only reproducible across captures when both the RF source and the
ADC sampling clock share a **10 MHz reference**.

In [ ]:
# ============================================================
# PHASE QUICK CHECK
# ============================================================
phases = []
freqs_peak = []

for m in measurements:
    x = np.loadtxt(m['file'], dtype=np.complex128)
    N = len(x)
    window = np.hanning(N)
    xw = x * window
    fft_c = np.fft.fftshift(np.fft.fft(xw) / N)
    freqs_fft = np.fft.fftshift(np.fft.fftfreq(N, 1/Fs))
    peak = np.argmax(np.abs(fft_c))
    phases.append(np.degrees(np.angle(fft_c[peak])))
    freqs_peak.append(freqs_fft[peak] / 1e6)

print(f"Phase across captures:")
print(f"  Mean:   {np.mean(phases):.1f}°")
print(f"  Std:    {np.std(phases):.1f}°")
print(f"  Range:  [{np.min(phases):.1f}°, {np.max(phases):.1f}°]")
print(f"  → Large std is expected (no phase lock between RF gen and ADC clock)")
print()
print(f"Tone frequency across captures:")
print(f"  Mean:   {np.mean(freqs_peak):.4f} MHz")
print(f"  Std:    {np.std(freqs_peak)*1e3:.2f} kHz")
print(f"  → Stable frequency confirms tone is correctly identified")